# SVM Ticket Classifier
Klasifikasi tiket IT helpdesk menggunakan **LinearSVC + TF-IDF** untuk memprediksi `category` dan `priority` berdasarkan kolom `description`.

**Input:** `cobacek.xlsx`  
**Output:** `cobacek_pred.xlsx`

## 1. Install Dependencies

In [ ]:
# !pip install pandas scikit-learn openpyxl

## 2. Import Libraries

In [ ]:
import pandas as pd
from openpyxl.comments import Comment
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

## 3. Load & Validasi Data

In [ ]:
INPUT_FILE = "cobacek.xlsx"

df = pd.read_excel(INPUT_FILE)
print(f"Total baris: {len(df)}")
print(f"Kolom: {df.columns.tolist()}")
df.head()

In [ ]:
# Validasi kolom wajib
for col in ["description", "category", "priority"]:
    if col not in df.columns:
        raise ValueError(f"Kolom '{col}' tidak ditemukan.")

df["description"] = df["description"].fillna("").astype(str)
print("Distribusi category:")
print(df["category"].value_counts())
print("\nDistribusi priority:")
print(df["priority"].value_counts())

## 4. Fungsi Training & Evaluasi SVM

In [ ]:
def train_and_evaluate(texts, labels):
    label_counts = labels.value_counts()
    stratify_labels = labels if label_counts.min() >= 2 else None

    x_train, x_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.2, random_state=42, stratify=stratify_labels
    )

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("svm", LinearSVC()),
    ])

    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)

    report = classification_report(y_test, y_pred, zero_division=0)
    report_dict = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    acc = accuracy_score(y_test, y_pred)

    return pipeline, report, report_dict, acc

## 5. Training Model Category & Priority

In [ ]:
category_model, category_report, category_report_dict, category_acc = train_and_evaluate(
    df["description"], df["category"]
)

priority_model, priority_report, priority_report_dict, priority_acc = train_and_evaluate(
    df["description"], df["priority"]
)

## 6. Hasil Evaluasi

In [ ]:
print(f"=== Category ===")
print(f"Accuracy: {round(category_acc, 4)}")
print(category_report)

In [ ]:
print(f"=== Priority ===")
print(f"Accuracy: {round(priority_acc, 4)}")
print(priority_report)

## 7. Retrain pada Full Data & Prediksi

In [ ]:
# Retrain pada seluruh data agar model lebih kuat saat dipakai prediksi
category_model.fit(df["description"], df["category"])
priority_model.fit(df["description"], df["priority"])

df["predicted_category"] = category_model.predict(df["description"])
df["predicted_priority"] = priority_model.predict(df["description"])
df["predicted_category_match"] = df["predicted_category"] == df["category"]
df["predicted_priority_match"] = df["predicted_priority"] == df["priority"]

print(f"Category match rate : {df['predicted_category_match'].mean():.2%}")
print(f"Priority match rate : {df['predicted_priority_match'].mean():.2%}")
df.head()

## 8. Metrics DataFrame

In [ ]:
category_metrics = (
    pd.DataFrame(category_report_dict)
    .T.loc[:, ["precision", "recall", "f1-score", "support"]]
    .rename_axis("Category Name")
)

priority_metrics = (
    pd.DataFrame(priority_report_dict)
    .T.loc[:, ["precision", "recall", "f1-score", "support"]]
    .rename_axis("Priority Name")
)

print("Category Metrics:")
display(category_metrics)
print("\nPriority Metrics:")
display(priority_metrics)

## 9. Export ke Excel

In [ ]:
prediction_notes_map = {
    "subject": "Judul ringkas tiket dari pengguna atau sistem",
    "description": "Detail masalah/kebutuhan; input utama model untuk prediksi",
    "answer": "Jawaban/solusi yang diberikan pada tiket",
    "type": "Tipe/klasifikasi tiket dari sumber data",
    "priority": "Label priority asli dari dataset (ground truth)",
    "category": "Label category asli dari dataset (ground truth)",
    "predicted_category": "Hasil prediksi category berbasis teks description",
    "predicted_priority": "Hasil prediksi priority berbasis teks description",
    "predicted_category_match": "True jika predicted_category sama dengan category",
    "predicted_priority_match": "True jika predicted_priority sama dengan priority",
}
prediction_notes = {
    index + 1: prediction_notes_map.get(col, "")
    for index, col in enumerate(df.columns)
}
metric_notes = {
    1: "Nama kelas/label yang dievaluasi",
    2: "Presisi per kelas: TP / (TP + FP)",
    3: "Recall per kelas: TP / (TP + FN)",
    4: "F1-score per kelas: 2 * (precision * recall) / (precision + recall)",
    5: "Jumlah data per kelas pada set evaluasi",
}

OUTPUT_FILE = "cobacek_pred.xlsx"
FALLBACK_FILE = "cobacek_pred_new.xlsx"

def write_output(file_path):
    with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Predictions")
        category_metrics.to_excel(writer, sheet_name="Category Accuracy")
        priority_metrics.to_excel(writer, sheet_name="Priority Accuracy")

        pred_sheet = writer.sheets["Predictions"]z
        for col_idx, note in prediction_notes.items():
            if note:
                pred_sheet.cell(row=1, column=col_idx).comment = Comment(note, "SVM")

        for sheet_name in ["Category Accuracy", "Priority Accuracy"]:
            sheet = writer.sheets[sheet_name]
            for col_idx, note in metric_notes.items():
                sheet.cell(row=1, column=col_idx).comment = Comment(note, "SVM")

try:
    write_output(OUTPUT_FILE)
    print(f"Output tersimpan: {OUTPUT_FILE}")
except PermissionError:
    write_output(FALLBACK_FILE)
    print(f"File terkunci, output tersimpan ke: {FALLBACK_FILE}")